# [DTAT] — Tech Challenge - Fase 3
## Análise do Mercado Brasileiro de Dados — State of Data Survey (2023–2025)

**Autores:** Tiago Antônio dos Santos, Isabele Cristina Felix Santos, Thais Marcondes Narbonne

Este notebook reproduz, a partir dos dados brutos das camadas **bronze / silver / gold** do
projeto, todos os números apresentados na apresentação executiva (HTML) e na documentação
("[DTAT] — Tech Challenge - Fase 3"). Os valores aqui calculados devem bater exatamente com
os dois artefatos citados.

**Metodologia de arredondamento:** para toda variável categórica que representa uma partição
completa da base (ex.: gênero, senioridade, região), os percentuais são calculados pelo
**método do maior resto** (largest remainder method), que garante que a soma dos percentuais
arredondados seja sempre exatamente 100,0% — evitando o efeito comum de arredondamentos
independentes que "quase fecham" (ex.: 99,6% ou 99,9%). Perguntas de múltipla escolha (onde
um respondente pode marcar mais de uma opção) não somam 100% por natureza e são tratadas à
parte, sempre com a base explícita de respondentes.

In [1]:
import glob
import math
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")

## 0. Localização dos dados e funções auxiliares

O notebook espera encontrar as pastas `bronze/`, `silver/` e `gold/` do projeto. Por padrão
ele procura a partir do diretório onde este notebook está salvo (e em `./data`, um nível
acima, e `../data`) — ajuste `DATA_DIR` manualmente abaixo se a estrutura do seu projeto for
diferente.

In [2]:
def resolve_data_dir() -> Path:
    candidates = [Path("."), Path("./data"), Path(".."), Path("../data")]
    for c in candidates:
        if (c / "gold").is_dir() and (c / "silver").is_dir():
            return c.resolve()
    raise FileNotFoundError(
        "Não encontrei as pastas 'gold' e 'silver'. Coloque este notebook na raiz do "
        "projeto (mesmo nível das pastas bronze/silver/gold) ou ajuste DATA_DIR manualmente "
        "na célula acima."
    )


DATA_DIR = resolve_data_dir()
GOLD = DATA_DIR / "gold"
SILVER = DATA_DIR / "silver"
print(f"Usando DATA_DIR = {DATA_DIR}")


def load_gold(table_name: str) -> pd.DataFrame:
    """Carrega uma tabela da camada gold (concatena arquivos-parte, se houver mais de um)."""
    files = sorted(glob.glob(str(GOLD / table_name / "*.csv")))
    if not files:
        raise FileNotFoundError(f"Nenhum CSV encontrado em gold/{table_name}/")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def load_silver_pesquisa(year: int, usecols=None) -> pd.DataFrame:
    """Carrega a pesquisa (wide format) da camada silver para um ano específico."""
    files = glob.glob(str(SILVER / "state_of_data" / f"pesquisa_{year}" / "*.csv"))
    if not files:
        raise FileNotFoundError(f"Nenhum CSV encontrado em silver/state_of_data/pesquisa_{year}/")
    if usecols is not None:
        return pd.read_csv(files[0], usecols=lambda c: c in usecols)
    return pd.read_csv(files[0])


def largest_remainder_pct(counts: dict, decimals: int = 1) -> dict:
    """
    Converte contagens absolutas em percentuais que somam exatamente 100,0%
    (método do maior resto / Hamilton apportionment).

    Cada valor é primeiro convertido em unidades de `10**decimals` (ex.: décimos de ponto
    percentual), arredondado para baixo; o resto de unidades necessário para fechar 100%
    é distribuído, uma unidade por vez, para as categorias com maior parte fracionária
    descartada. Isso é o que garante soma == 100,0 sem viesar sistematicamente uma
    categoria específica.
    """
    total = sum(counts.values())
    if total == 0:
        return {k: 0.0 for k in counts}
    scale = 10**decimals
    target_units = 100 * scale
    raw_units = {k: v / total * target_units for k, v in counts.items()}
    floor_units = {k: math.floor(v) for k, v in raw_units.items()}
    remainder = int(round(target_units - sum(floor_units.values())))
    order = sorted(counts.keys(), key=lambda k: -(raw_units[k] - floor_units[k]))
    for k in order[:remainder]:
        floor_units[k] += 1
    return {k: floor_units[k] / scale for k in counts}


def pct_table(counts: pd.Series, label_col: str, decimals: int = 1) -> pd.DataFrame:
    """Monta um DataFrame (categoria, n, %) usando o método do maior resto."""
    counts_dict = counts.to_dict()
    pcts = largest_remainder_pct(counts_dict, decimals=decimals)
    df = pd.DataFrame(
        {label_col: list(counts_dict.keys()), "n": list(counts_dict.values())}
    )
    df["%"] = df[label_col].map(pcts)
    df = df.sort_values("n", ascending=False).reset_index(drop=True)
    return df


def assert_sums_100(df: pd.DataFrame, name: str, pct_col: str = "%"):
    total = round(df[pct_col].sum(), 6)
    status = "OK" if abs(total - 100.0) < 1e-6 else "FALHOU"
    print(f"[checagem] soma de '{name}' = {total}%  -> {status}")
    assert abs(total - 100.0) < 1e-6, f"{name} não soma 100% (soma={total})"

Usando DATA_DIR = /state-of-data-survey-analysis


## 1. Carregamento das tabelas principais (camada gold)

A tabela fato `tbl_fato_perfil_profissional` é a base de praticamente toda a análise
(um registro por respondente/ano). O salário médio de cada respondente é obtido via *join*
com a dimensão `tbl_dimensao_faixa_salarial` (usando o ponto médio de cada faixa salarial
declarada).

In [3]:
perfil = load_gold("tbl_fato_perfil_profissional")
tecnologias = load_gold("tbl_fato_tecnologias")
ia_generativa = load_gold("tbl_fato_ia_generativa")
gestao = load_gold("tbl_fato_gestao")
satisfacao = load_gold("tbl_fato_satisfacao")
faixa_salarial = load_gold("tbl_dimensao_faixa_salarial")

# A dimensão de faixa salarial tem uma linha duplicada/corrigida para uma das faixas
# (valor_corrigido_da_fonte == True); mantemos apenas a versão original (False) para o join.
faixa_salarial_clean = (
    faixa_salarial[faixa_salarial["valor_corrigido_da_fonte"] == False]
    .drop_duplicates(subset=["faixa_salarial"])
)
mapa_salario = dict(zip(faixa_salarial_clean["faixa_salarial"], faixa_salarial_clean["valor_medio"]))
perfil["salario_medio"] = perfil["faixa_salarial"].map(mapa_salario)

TOTAL_RESPONDENTES = len(perfil)
print(f"Total de respondentes (2023-2025): {TOTAL_RESPONDENTES}")
print(perfil["ano_pesquisa"].value_counts().sort_index())

Total de respondentes (2023-2025): 14002
ano_pesquisa
2023    5293
2024    5215
2025    3494
Name: count, dtype: int64


## Pergunta 1 — Como está estruturado o mercado brasileiro de Dados?

### 1.1 Setor de atuação

Para manter a tabela legível, os 6 setores mais frequentes são exibidos individualmente e
o restante é agrupado em "Outros" — mas o percentual usa a base completa de respondentes que
informaram o setor (nenhum respondente é descartado do cálculo).

In [4]:
setor_counts_full = perfil["setor_atual"].value_counts()
# "Outra Opção" é a própria resposta de catch-all do questionário (não nomeia um setor real);
# ela é tratada como parte do balde residual "Outros", não como um dos 6 setores nomeados,
# mesmo tendo volume maior que alguns setores nomeados (ex.: Indústria, Educação).
setor_counts_nomeados = setor_counts_full.drop(index="Outra Opção", errors="ignore")
top6_setor = setor_counts_nomeados.head(6)
total_setor_respondido = int(perfil["setor_atual"].notna().sum())
outros_setor_n = int(total_setor_respondido - top6_setor.sum())

setor_counts = top6_setor.to_dict()
setor_counts["Outros"] = outros_setor_n

df_setor = pct_table(pd.Series(setor_counts), "setor")
assert_sums_100(df_setor, "1.1 Setor de atuação")
print(f"Base: {total_setor_respondido} respondentes")
df_setor

[checagem] soma de '1.1 Setor de atuação' = 100.0%  -> OK
Base: 12841 respondentes


,setor,n,%
0,Outros,4496,35.0
1,Finanças ou Bancos,2560,19.9
2,Tecnologia/Fábrica de Software,2361,18.4
3,Área de Consultoria,1027,8.0
4,Varejo,953,7.4
5,Indústria,878,6.9
6,Educação,566,4.4


### 1.2 Porte das empresas empregadoras

`num_funcionarios` tem 2 registros com um valor inconsistente na base ("de 501 a 100",
provável erro de digitação/ETL na fonte — não corresponde a nenhuma das 8 faixas oficiais do
questionário). Esses 2 registros são excluídos explicitamente da tabela abaixo, e não de
forma silenciosa.

In [5]:
FAIXAS_PORTE_VALIDAS = [
    "de 1 a 5", "de 6 a 10", "de 11 a 50", "de 51 a 100",
    "de 101 a 500", "de 501 a 1.000", "de 1.001 a 3.000", "Acima de 3.000",
]
porte_counts_full = perfil["num_funcionarios"].value_counts()
n_invalidos = int(porte_counts_full.drop(index=FAIXAS_PORTE_VALIDAS, errors="ignore").sum())
if n_invalidos:
    print(f"[nota de qualidade de dados] {n_invalidos} registro(s) com valor de "
          f"'num_funcionarios' fora das 8 faixas oficiais foram excluídos: "
          f"{porte_counts_full.drop(index=FAIXAS_PORTE_VALIDAS, errors='ignore').to_dict()}")

porte_counts = porte_counts_full.reindex(FAIXAS_PORTE_VALIDAS).dropna().astype(int)
df_porte = pct_table(porte_counts, "porte")
assert_sums_100(df_porte, "1.2 Porte das empresas")
print(f"Base: {int(porte_counts.sum())} respondentes")
df_porte

[nota de qualidade de dados] 2 registro(s) com valor de 'num_funcionarios' fora das 8 faixas oficiais foram excluídos: {'de 501 a 100': 2}
[checagem] soma de '1.2 Porte das empresas' = 100.0%  -> OK
Base: 12839 respondentes


,porte,n,%
0,Acima de 3.000,5730,44.6
1,de 101 a 500,2121,16.5
2,de 1.001 a 3.000,1445,11.3
3,de 501 a 1.000,1231,9.6
4,de 51 a 100,905,7.1
5,de 11 a 50,865,6.7
6,de 1 a 5,332,2.6
7,de 6 a 10,210,1.6


### 1.3 Distribuição regional

In [6]:
regiao_counts = perfil["regiao_atual"].value_counts()
df_regiao = pct_table(regiao_counts, "regiao")
assert_sums_100(df_regiao, "1.3 Distribuição regional")
df_regiao

[checagem] soma de '1.3 Distribuição regional' = 100.0%  -> OK


,regiao,n,%
0,Sudeste,8474,62.3
1,Sul,2531,18.6
2,Nordeste,1504,11.0
3,Centro-oeste,907,6.7
4,Norte,195,1.4


### 1.4 Distribuição por senioridade

In [7]:
senioridade_counts = perfil["senioridade"].value_counts()
df_senioridade = pct_table(senioridade_counts, "senioridade")
assert_sums_100(df_senioridade, "1.4 Distribuição por senioridade")
df_senioridade

[checagem] soma de '1.4 Distribuição por senioridade' = 100.0%  -> OK


,senioridade,n,%
0,Sênior,3849,37.9
1,Pleno,3543,34.8
2,Júnior,2432,23.9
3,Especialista/Staff+,349,3.4


### 1.5 Distribuição por gênero

In [8]:
genero_counts = perfil["genero"].value_counts()
df_genero = pct_table(genero_counts, "genero")
assert_sums_100(df_genero, "1.5 Distribuição por gênero")
df_genero

[checagem] soma de '1.5 Distribuição por gênero' = 100.0%  -> OK


,genero,n,%
0,Masculino,10649,76.0
1,Feminino,3285,23.5
2,Prefiro não informar,44,0.3
3,Outro,24,0.2


### 1.6 Modelo de trabalho

`modalidade_atual` tem 2 variações de híbrido no questionário original ("flexível" e
"dias fixos"); ambas são agrupadas em uma única categoria "Híbrido" para leitura executiva.

In [9]:
def simplifica_modalidade(m):
    if pd.isna(m):
        return None
    m_low = m.lower()
    if "remoto" in m_low:
        return "Remoto"
    if "híbrid" in m_low:
        return "Híbrido"
    if "presencial" in m_low:
        return "Presencial"
    return "Outro"

perfil["modalidade_simples"] = perfil["modalidade_atual"].apply(simplifica_modalidade)
modelo_counts = perfil["modalidade_simples"].value_counts()
df_modelo = pct_table(modelo_counts, "modelo")
assert_sums_100(df_modelo, "1.6 Modelo de trabalho")
df_modelo

[checagem] soma de '1.6 Modelo de trabalho' = 100.0%  -> OK


,modelo,n,%
0,Remoto,5703,44.4
1,Híbrido,4886,38.1
2,Presencial,2252,17.5


### 1.x (complementar) Raça/cor autodeclarada e PCD

Não é uma das perguntas centrais da Seção 1 na apresentação, mas é citada na Seção 3
(diversidade) — computada aqui pela mesma metodologia para reaproveitamento.

In [10]:
raca_counts = perfil["cor_raca_etnia"].value_counts()
df_raca = pct_table(raca_counts, "raca_cor")
assert_sums_100(df_raca, "Raça/cor autodeclarada")
df_raca

[checagem] soma de 'Raça/cor autodeclarada' = 100.0%  -> OK


,raca_cor,n,%
0,Branca,9163,65.5
1,Parda,3302,23.6
2,Preta,956,6.8
3,Amarela,419,3.0
4,Prefiro não informar,99,0.7
5,Outra,35,0.2
6,Indígena,26,0.2


In [11]:
pcd_counts = perfil["pcd"].value_counts()
df_pcd = pct_table(pcd_counts, "pcd")
assert_sums_100(df_pcd, "PCD")
df_pcd

[checagem] soma de 'PCD' = 100.0%  -> OK


,pcd,n,%
0,Não,13527,96.6
1,Sim,390,2.8
2,Prefiro não informar,85,0.6


### Síntese da Seção 1

In [12]:
pct_setor_dict = dict(zip(df_setor["setor"], df_setor["%"]))
pct_porte_dict = dict(zip(df_porte["porte"], df_porte["%"]))
pct_regiao_dict = dict(zip(df_regiao["regiao"], df_regiao["%"]))
pct_senioridade_dict = dict(zip(df_senioridade["senioridade"], df_senioridade["%"]))
pct_genero_dict = dict(zip(df_genero["genero"], df_genero["%"]))
pct_modelo_dict = dict(zip(df_modelo["modelo"], df_modelo["%"]))

pleno_senior = pct_senioridade_dict["Pleno"] + pct_senioridade_dict["Sênior"]
remoto_hibrido = pct_modelo_dict["Remoto"] + pct_modelo_dict["Híbrido"]

# "Outros" é o maior balde em volume por agregar dezenas de setores residuais — não é um
# setor de fato, então é excluído ao escolher os 2 setores nominais mais frequentes.
top2_setores_nominais = df_setor[df_setor["setor"] != "Outros"].head(2)["setor"].tolist()

print(
    f"O profissional de Dados \"típico\" no Brasil trabalha em uma grande empresa "
    f"({pct_porte_dict['Acima de 3.000']:.1f}% em corporações com 3.000+ funcionários) dos "
    f"setores de {top2_setores_nominais[0]} ({pct_setor_dict[top2_setores_nominais[0]]:.1f}%) "
    f"ou {top2_setores_nominais[1]} ({pct_setor_dict[top2_setores_nominais[1]]:.1f}%), está "
    f"no {df_regiao.iloc[0]['regiao']} ({pct_regiao_dict[df_regiao.iloc[0]['regiao']]:.1f}%), "
    f"tem senioridade Pleno/Sênior ({pleno_senior:.1f}% somados) e atua em regime remoto ou "
    f"híbrido ({remoto_hibrido:.1f}% somados). É majoritariamente homem "
    f"({pct_genero_dict['Masculino']:.1f}%)."
)

O profissional de Dados "típico" no Brasil trabalha em uma grande empresa (44.6% em corporações com 3.000+ funcionários) dos setores de Finanças ou Bancos (19.9%) ou Tecnologia/Fábrica de Software (18.4%), está no Sudeste (62.3%), tem senioridade Pleno/Sênior (72.7% somados) e atua em regime remoto ou híbrido (82.5% somados). É majoritariamente homem (76.0%).


## Pergunta 2 — Quais perfis profissionais são mais valorizados pelo mercado?

O dataset não tem número de vagas em aberto por cargo — "valorização" é inferida por dois
ângulos: **volume** de profissionais empregados em cada cargo e **salário médio** de mercado.

### 2.1 Cargos com maior volume no mercado

`cargo_atual` não é respondido por todos: dos 14.002 respondentes totais, 3.829 não
preencheram esse campo. A base de referência para os percentuais abaixo é, portanto, os
**10.173 respondentes que informaram um cargo atual** — não os 14.002 totais.

In [13]:
cargo_counts = perfil["cargo_atual"].value_counts()
BASE_CARGO = int(cargo_counts.sum())
print(f"Base (respondentes que informaram cargo_atual): {BASE_CARGO}")

df_cargo_volume = cargo_counts.head(10).rename_axis("cargo").reset_index(name="n")
df_cargo_volume["%"] = (df_cargo_volume["n"] / BASE_CARGO * 100).round(1)
df_cargo_volume

Base (respondentes que informaram cargo_atual): 10173


,cargo,n,%
0,Analista de Dados/Data Analyst,2463,24.2
1,Cientista de Dados/Data Scientist,1797,17.7
2,Analista de BI/BI Analyst,1117,11.0
3,Engenheiro de Dados/Data Engineer/Data Architect,1014,10.0
4,Outra Opção,722,7.1
5,Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect,684,6.7
6,Analista de Negócios/Business Analyst,520,5.1
7,Analytics Engineer,502,4.9
8,Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas,333,3.3
9,Engenheiro de Machine Learning/ML Engineer/AI Engineer,284,2.8


In [14]:
analista_cientista_n = int(
    cargo_counts.get("Analista de Dados/Data Analyst", 0)
    + cargo_counts.get("Cientista de Dados/Data Scientist", 0)
)
pct_analista_cientista = round(analista_cientista_n / BASE_CARGO * 100, 1)
print(
    f"Analistas de Dados e Cientistas de Dados somados: {analista_cientista_n} "
    f"({pct_analista_cientista}% dos {BASE_CARGO} respondentes que informaram o cargo atual)."
)

Analistas de Dados e Cientistas de Dados somados: 4260 (41.9% dos 10173 respondentes que informaram o cargo atual).


### 2.2 Cargos mais bem remunerados (salário médio mensal)

Tabela completa dos cargos com 30+ respondentes, ordenada por salário médio — a apresentação
e a documentação destacam os primeiros desta lista.

In [15]:
cargo_salario = (
    perfil.groupby("cargo_atual")["salario_medio"]
    .agg(salario_medio="mean", n="count")
    .reset_index()
)
cargo_salario = cargo_salario[cargo_salario["n"] >= 30].sort_values("salario_medio", ascending=False)
cargo_salario["salario_medio"] = cargo_salario["salario_medio"].round(0)
cargo_salario.reset_index(drop=True)

,cargo_atual,salario_medio,n
0,Arquiteto de Dados/Data Architect,"16,553.0",76
1,Engenheiro de Machine Learning/ML Engineer/AI Engineer,"15,886.0",284
2,Engenheiro de Dados/Data Engineer/Data Architect,"13,264.0",1014
3,Data Product Manager/ Product Manager (PM/APM/DPM/GPM/PO),"12,665.0",194
4,Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect,"11,586.0",684
5,Analytics Engineer,"11,575.0",502
6,Cientista de Dados/Data Scientist,"11,572.0",1797
7,Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas,"10,555.0",333
8,Outras Engenharias (não inclui dev),"10,128.0",86
9,Professor/Pesquisador,"10,124.0",65


### 2.3 Senioridade: distribuição e remuneração

In [16]:
senioridade_salario = (
    perfil.groupby("senioridade")["salario_medio"]
    .agg(salario_medio="mean", n="count")
    .dropna()
    .reset_index()
)
senioridade_salario["salario_medio"] = senioridade_salario["salario_medio"].round(0)
senioridade_salario = senioridade_salario.merge(
    df_senioridade[["senioridade", "%"]], on="senioridade"
).sort_values("salario_medio", ascending=False)
senioridade_salario.reset_index(drop=True)

,senioridade,salario_medio,n,%
0,Especialista/Staff+,"19,685.0",349,3.4
1,Sênior,"14,117.0",3849,37.9
2,Pleno,"7,898.0",3543,34.8
3,Júnior,"4,048.0",2432,23.9


## Pergunta 3 — Qual é o cenário de diversidade de gênero nas carreiras de dados?

### 3.1 Composição geral e evolução (2023-2025)

In [17]:
df_genero  # reaproveitando a tabela calculada na Seção 1.5

,genero,n,%
0,Masculino,10649,76.0
1,Feminino,3285,23.5
2,Prefiro não informar,44,0.3
3,Outro,24,0.2


In [18]:
genero_evolucao = {}
for ano, grupo in perfil.groupby("ano_pesquisa"):
    pct_ano = largest_remainder_pct(grupo["genero"].value_counts().to_dict())
    genero_evolucao[int(ano)] = pct_ano
    assert abs(sum(pct_ano.values()) - 100.0) < 1e-6, f"Ano {ano} não soma 100%"

df_genero_evolucao = pd.DataFrame(genero_evolucao).T
df_genero_evolucao.index.name = "ano"
print("[checagem] soma por ano:")
print(df_genero_evolucao.sum(axis=1))
df_genero_evolucao

[checagem] soma por ano:
ano
2023   100.0
2024   100.0
2025   100.0
dtype: float64


,Masculino,Feminino,Prefiro não informar,Outro
ano,,,,
2023,75.1,24.4,0.3,0.2
2024,76.1,23.5,0.3,0.1
2025,77.5,21.9,0.4,0.2


In [19]:
fem_2023 = genero_evolucao[2023]["Feminino"]
fem_2025 = genero_evolucao[2025]["Feminino"]
print(
    f"Participação feminina: de {fem_2023:.1f}% (2023) para {fem_2025:.1f}% (2025) "
    f"-> variação de {fem_2025 - fem_2023:+.1f} pontos percentuais."
)

Participação feminina: de 24.4% (2023) para 21.9% (2025) -> variação de -2.5 pontos percentuais.


### 3.2 Gênero, senioridade e remuneração

In [20]:
genero_por_senioridade = {}
for sen, grupo in perfil.dropna(subset=["senioridade"]).groupby("senioridade"):
    genero_por_senioridade[sen] = largest_remainder_pct(grupo["genero"].value_counts().to_dict())

df_genero_senioridade = pd.DataFrame(genero_por_senioridade).T
df_genero_senioridade = df_genero_senioridade.reindex(["Júnior", "Pleno", "Sênior", "Especialista/Staff+"])
print("[checagem] soma por senioridade:")
print(df_genero_senioridade.sum(axis=1))
df_genero_senioridade

[checagem] soma por senioridade:
Júnior                100.0
Pleno                 100.0
Sênior                100.0
Especialista/Staff+   100.0
dtype: float64


,Masculino,Feminino,Prefiro não informar,Outro
Júnior,71.3,28.2,0.4,0.1
Pleno,74.8,24.7,0.3,0.2
Sênior,78.2,21.3,0.3,0.2
Especialista/Staff+,79.1,20.0,0.6,0.3


In [21]:
salario_por_genero = (
    perfil[perfil["genero"].isin(["Masculino", "Feminino"])]
    .groupby("genero")["salario_medio"]
    .agg(salario_medio="mean", n="count")
)
salario_por_genero["salario_medio"] = salario_por_genero["salario_medio"].round(0)
gap_pct = round(
    100
    * (1 - salario_por_genero.loc["Feminino", "salario_medio"] / salario_por_genero.loc["Masculino", "salario_medio"]),
    1,
)
print(f"Gap salarial (mulheres ganham, em média, {gap_pct}% menos que homens):")
salario_por_genero

Gap salarial (mulheres ganham, em média, 18.6% menos que homens):


,salario_medio,n
genero,,
Feminino,"9,908.0",2953
Masculino,"12,165.0",9830


### 3.3 Percepção de carreira prejudicada por gênero e raça (camada silver)

Esta pergunta específica (multiescolha: "quais aspectos da sua carreira você sente que
foram prejudicados") **não está materializada na camada gold** como um recorte por gênero ou
raça — os arquivos-fato do gold trazem apenas o aspecto prejudicado (ex.: "velocidade de
progressão"), sem a atribuição de motivo. Para responder a essa pergunta específica é
necessário usar a camada **silver**, que preserva as colunas originais do questionário
(`profissao_prejudicada_genero`, `profissao_prejudicada_etnia`, `profissao_prejudicada_pcd`).
Isso é declarado aqui explicitamente, conforme a regra de zero alucinação do projeto.

In [22]:
COLS_PREJUDICADA = [
    "genero",
    "profissao_prejudicada",
    "profissao_prejudicada_genero",
    "profissao_prejudicada_etnia",
    "profissao_prejudicada_pcd",
]
harmed_frames = []
for ano in (2023, 2024, 2025):
    df_ano = load_silver_pesquisa(ano, usecols=COLS_PREJUDICADA)
    df_ano["ano_pesquisa"] = ano
    harmed_frames.append(df_ano)
harmed = pd.concat(harmed_frames, ignore_index=True)

sub_cols = ["profissao_prejudicada_genero", "profissao_prejudicada_etnia", "profissao_prejudicada_pcd"]
respondeu_pergunta = harmed[harmed[sub_cols].notna().any(axis=1)]

pct_genero_motivo = round(100 * respondeu_pergunta["profissao_prejudicada_genero"].fillna(0).sum() / len(respondeu_pergunta), 1)
pct_etnia_motivo = round(100 * respondeu_pergunta["profissao_prejudicada_etnia"].fillna(0).sum() / len(respondeu_pergunta), 1)
pct_pcd_motivo = round(100 * respondeu_pergunta["profissao_prejudicada_pcd"].fillna(0).sum() / len(respondeu_pergunta), 1)

print(f"Base (respondentes que responderam a esta pergunta na camada silver): {len(respondeu_pergunta)}")
print(f"Apontaram GÊNERO como motivo de carreira prejudicada: {pct_genero_motivo}%")
print(f"Apontaram ETNIA/RAÇA como motivo: {pct_etnia_motivo}%")
print(f"Apontaram PCD como motivo: {pct_pcd_motivo}%")

Base (respondentes que responderam a esta pergunta na camada silver): 7140
Apontaram GÊNERO como motivo de carreira prejudicada: 30.3%
Apontaram ETNIA/RAÇA como motivo: 21.4%
Apontaram PCD como motivo: 3.2%


In [23]:
por_genero_do_respondente = {}
for g in ["Masculino", "Feminino"]:
    sub = respondeu_pergunta[respondeu_pergunta["genero"] == g]
    por_genero_do_respondente[g] = {
        "n_respondentes": int(len(sub)),
        "pct_prejudicado_por_genero": round(100 * sub["profissao_prejudicada_genero"].fillna(0).sum() / len(sub), 1),
    }
pd.DataFrame(por_genero_do_respondente).T

,n_respondentes,pct_prejudicado_por_genero
Masculino,"3,810.0",3.8
Feminino,"3,262.0",60.9


## Pergunta 4 — Quais tecnologias apresentam maior adoção entre os profissionais?

Cada bloco de tecnologia (linguagens, bancos de dados, ferramentas de BI, técnicas de
ciência de dados) é uma pergunta de **múltipla escolha** — o percentual de cada item é
calculado sobre os respondentes únicos que responderam àquele bloco específico (não sobre
os 14.002 totais), e por isso os percentuais de um mesmo bloco não somam 100%.

In [24]:
def top_tecnologia(categoria: str, n: int = 10) -> pd.DataFrame:
    sub = tecnologias[tecnologias["categoria"] == categoria]
    base = sub["respondente_id"].nunique()
    vc = sub["tecnologia"].value_counts().head(n)
    df = vc.rename_axis("item").reset_index(name="n")
    df["% (base=" + str(base) + ")"] = (df["n"] / base * 100).round(1)
    return df

### 4.1 Linguagens de programação

In [25]:
df_linguagens = top_tecnologia("linguagem", 6)
df_linguagens

,item,n,% (base=9007)
0,SQL,8064,89.5
1,Python,7687,85.3
2,R,1087,12.1
3,Java,646,7.2
4,JavaScript,505,5.6
5,Visual Basic/VBA,501,5.6


### 4.2 Bancos de dados e armazenamento

In [26]:
df_bancos = top_tecnologia("banco_dados", 7)
df_bancos

,item,n,% (base=9289)
0,PostgreSQL,2965,31.9
1,SQL Server,2963,31.9
2,Databricks,2612,28.1
3,Google BigQuery,2490,26.8
4,MySQL,2357,25.4
5,Amazon S3,2241,24.1
6,Oracle,1739,18.7


### 4.3 Ferramentas de BI

In [27]:
df_bi = top_tecnologia("ferramenta_bi", 6)
df_bi

,item,n,% (base=8288)
0,Microsoft Power BI,5392,65.1
1,Looker,2204,26.6
2,Tableau,1687,20.4
3,Looker Studio,1584,19.1
4,Excel/Google Sheets,992,12.0
5,Metabase,819,9.9


### 4.4 Cloud preferida

`servico_cloud_preferido` tem muitas respostas de texto livre residuais e ambíguas (ex.:
"não sei", "sem preferência", nomes de ferramentas específicas com n=1) — o percentual
abaixo é calculado apenas sobre as 3 opções com volume relevante (AWS, GCP, Azure).

In [28]:
cloud_counts_full = perfil["servico_cloud_preferido"].value_counts()
cloud_top3 = cloud_counts_full.head(3)
print(f"Excluídas {int(cloud_counts_full.iloc[3:].sum())} respostas residuais/ambíguas de "
      f"{len(cloud_counts_full) - 3} categorias diferentes.")
df_cloud = cloud_top3.rename_axis("servico").reset_index(name="n")
df_cloud["% (entre AWS/GCP/Azure)"] = (df_cloud["n"] / cloud_top3.sum() * 100).round(1)
df_cloud

Excluídas 1933 respostas residuais/ambíguas de 52 categorias diferentes.


,servico,n,% (entre AWS/GCP/Azure)
0,Amazon Web Services (AWS),3453,45.9
1,Google Cloud (GCP),2178,29.0
2,Azure (Microsoft),1889,25.1


### 4.5 Técnicas e ferramentas de Ciência de Dados

In [29]:
df_tecnicas_cientista = top_tecnologia("tecnica_cientista", 6)
df_tecnicas_cientista

,item,n,% (base=2010)
0,Modelos de regressão,1410,70.1
1,Redes neurais/árvore,1191,59.3
2,Estatística clássica,1097,54.6
3,Clusterização,1068,53.1
4,Séries temporais,798,39.7
5,LLMs para negócio,710,35.3


## Pergunta 5 — Qual é o índice de adoção de Inteligência Artificial e seu impacto?

### 5.1 Adoção pessoal de IA generativa — evolução 2023-2025

A base de respondentes deste bloco varia por ano (nem todos os respondentes de cada edição
responderam a esta pergunta) — o percentual "não uso" é calculado sobre quem respondeu ao
bloco naquele ano especificamente, não sobre o total de respondentes do ano.

In [30]:
uso_pessoal = ia_generativa[ia_generativa["categoria"] == "uso_pessoal"]

tendencia = {}
for ano, grupo in uso_pessoal.groupby("ano_pesquisa"):
    base_ano = grupo["respondente_id"].nunique()
    nao_uso_n = (grupo["item"] == "Não uso IA para produtividade").sum()
    tendencia[int(ano)] = {
        "base": int(base_ano),
        "% não uso": round(100 * nao_uso_n / base_ano, 1),
        "% usa alguma solução": round(100 * (base_ano - nao_uso_n) / base_ano, 1),
    }
df_tendencia_ia = pd.DataFrame(tendencia).T
df_tendencia_ia.index.name = "ano"
df_tendencia_ia

,base,% não uso,% usa alguma solução
ano,,,
2023,"3,772.0",19.7,80.3
2024,"3,617.0",6.5,93.5
2025,"2,105.0",2.1,97.9


### Uso pessoal de IA — "entre quem usa" (base corrigida)

**Atenção metodológica:** o bloco `uso_pessoal` tem 9.494 respondentes únicos no total, mas
1.024 deles marcaram exclusivamente "Não uso IA para produtividade". O percentual de cada
forma de uso (gratuita, Copilot, paga pela empresa, paga do bolso) deve ser calculado sobre
quem **de fato usa** IA — não sobre o bloco inteiro (o que sub-representaria os padrões de
uso real). A base de usuários reais inclui também os poucos respondentes (44) que, de forma
inconsistente na própria pesquisa, marcaram "não uso" e ainda assim relataram uma forma de
uso real — esses são contados como usuários.

In [31]:
nao_uso_ids = set(uso_pessoal.loc[uso_pessoal["item"] == "Não uso IA para produtividade", "respondente_id"])
todos_ids = set(uso_pessoal["respondente_id"])
usa_algo_ids = set(uso_pessoal.loc[uso_pessoal["item"] != "Não uso IA para produtividade", "respondente_id"])

BASE_BLOCO = len(todos_ids)
BASE_USUARIOS_REAIS = len(usa_algo_ids)
print(f"Base do bloco (todos que responderam): {BASE_BLOCO}")
print(f"Marcaram 'não uso' (exclusivamente ou não): {len(nao_uso_ids)}")
print(f"Usuários reais de IA (marcaram ao menos 1 uso real): {BASE_USUARIOS_REAIS}")

uso_pessoal_real = uso_pessoal[uso_pessoal["item"] != "Não uso IA para produtividade"]
vc_uso = uso_pessoal_real["item"].value_counts()
df_uso_pessoal = vc_uso.rename_axis("item").reset_index(name="n")
df_uso_pessoal["% (entre usuários reais)"] = (df_uso_pessoal["n"] / BASE_USUARIOS_REAIS * 100).round(1)
df_uso_pessoal

Base do bloco (todos que responderam): 9494
Marcaram 'não uso' (exclusivamente ou não): 1024
Usuários reais de IA (marcaram ao menos 1 uso real): 8514


,item,n,% (entre usuários reais)
0,Uso solução gratuita,4987,58.6
1,Uso solução tipo Copilot,1930,22.7
2,Empresa paga a solução,1828,21.5
3,Uso e pago (próprio bolso),1390,16.3


### 5.2 Uso de IA no ambiente de trabalho (segunda pergunta do bloco de IA generativa)

**Nota:** diferente da Seção 5.3 (`ai_prioridade`, respondida exclusivamente por gestores —
verificado abaixo via `atua_como_gestor`), o campo `uso_empresa_gestor` da camada gold **não
está restrito a esse grupo**: ao cruzar com `atua_como_gestor`, todos os 9.494 respondentes
deste bloco são, na verdade, gestores == 0/FALSE. Por isso o resultado é reportado de forma
agregada, sem atribuir a visão a um subgrupo específico — o nome da coluna na fonte é
enganoso e não deve ser tomado como "visão de gestores".

In [32]:
uso_trabalho = ia_generativa[ia_generativa["categoria"] == "uso_empresa_gestor"]
ids_uso_trabalho = uso_trabalho["respondente_id"].unique()
perfil_uso_trabalho = perfil[perfil["respondente_id"].isin(ids_uso_trabalho)]
print("Composição de atua_como_gestor entre os respondentes deste bloco (mostra que NÃO é exclusivo de gestores):")
print(perfil_uso_trabalho["atua_como_gestor"].value_counts())

base_uso_trabalho = uso_trabalho["respondente_id"].nunique()
vc_uso_trabalho = uso_trabalho["item"].value_counts()
df_uso_trabalho = vc_uso_trabalho.rename_axis("item").reset_index(name="n")
df_uso_trabalho["% (base=" + str(base_uso_trabalho) + ")"] = (df_uso_trabalho["n"] / base_uso_trabalho * 100).round(1)
df_uso_trabalho

Composição de atua_como_gestor entre os respondentes deste bloco (mostra que NÃO é exclusivo de gestores):
atua_como_gestor
0        5877
FALSE    3617
Name: count, dtype: int64


,item,n,% (base=9494)
0,Uso independente/descentralizado,4550,47.9
1,Melhorar produtos externos,2450,25.8
2,Melhorar produtos internos,2234,23.5
3,Devs usando Copilots,2209,23.3
4,Direcionamento centralizado,1911,20.1
5,Não é prioridade,1336,14.1
6,Não sei opinar,832,8.8
7,Principal frente do negócio,598,6.3


### 5.3 Prioridade estratégica de IA nas empresas

`ai_prioridade` (perfil) — verificado abaixo que é respondido exclusivamente por gestores
(`atua_como_gestor` == 1/TRUE em 100% dos casos).

In [33]:
respondeu_prioridade = perfil[perfil["ai_prioridade"].notna()]
print("Composição de atua_como_gestor entre quem respondeu ai_prioridade:")
print(respondeu_prioridade["atua_como_gestor"].value_counts())

df_prioridade = pct_table(respondeu_prioridade["ai_prioridade"].value_counts(), "resposta")
assert_sums_100(df_prioridade, "5.3 Prioridade estratégica de IA")
df_prioridade

Composição de atua_como_gestor entre quem respondeu ai_prioridade:
atua_como_gestor
1       1548
TRUE    1045
Name: count, dtype: int64
[checagem] soma de '5.3 Prioridade estratégica de IA' = 100.0%  -> OK


,resposta,n,%
0,"Sim, está entre nossas principais prioridades para os próximos 2-4 anos (com...",779,30.0
1,"Sim, é nossa principal prioridade como empresa (com foco executivo significa...",500,19.3
2,Não é uma iniciativa que estamos focando e não tem sido uma prioridade.,487,18.8
3,"Mais ou menos... É uma das várias iniciativas que estamos impulsionando, mas...",470,18.1
4,"Mais ou menos... É uma das várias iniciativas que estamos impulsionando, mas...",275,10.6
5,Não sei opinar sobre esse assunto.,82,3.2


### 5.4 Principais barreiras à adoção (motivos de não uso)

In [34]:
motivo_desuso = ia_generativa[ia_generativa["categoria"] == "motivo_desuso"]
base_desuso = motivo_desuso["respondente_id"].nunique()
vc_desuso = motivo_desuso["item"].value_counts().head(8)
df_motivos_desuso = vc_desuso.rename_axis("item").reset_index(name="n")
df_motivos_desuso["% (base=" + str(base_desuso) + ")"] = (df_motivos_desuso["n"] / base_desuso * 100).round(1)
df_motivos_desuso

,item,n,% (base=2290)
0,Falta de expertise/recursos,842,36.8
1,Falta de casos de uso claros,825,36.0
2,Dados não prontos pra IA,757,33.1
3,Preocupação com segurança/privacidade,739,32.3
4,ROI não comprovado,542,23.7
5,Falta de confiabilidade (alucinação),402,17.6
6,Incerteza regulatória,301,13.1
7,Preocupação com propriedade intelectual,298,13.0


## Pergunta 6 — Existem diferenças relevantes entre regiões, senioridades ou modelos de trabalho?

### 6.1 Diferenças regionais

In [35]:
salario_regiao = (
    perfil.groupby("regiao_atual")["salario_medio"].agg(salario_medio="mean", n="count").dropna()
)
salario_regiao = salario_regiao[salario_regiao["n"] >= 20].sort_values("salario_medio", ascending=False)
salario_regiao["salario_medio"] = salario_regiao["salario_medio"].round(0)
df_salario_regiao = salario_regiao.reset_index().merge(
    df_regiao[["regiao", "%"]], left_on="regiao_atual", right_on="regiao"
).drop(columns="regiao").rename(columns={"regiao_atual": "regiao"})
df_salario_regiao

,regiao,salario_medio,n,%
0,Sudeste,"12,174.0",7852,62.3
1,Centro-oeste,"10,876.0",825,6.7
2,Sul,"10,412.0",2384,18.6
3,Norte,"9,176.0",162,1.4
4,Nordeste,"9,159.0",1277,11.0


### 6.2 Diferenças por senioridade

Já detalhado na Seção 2.3 — reaproveitado aqui para referência cruzada.

In [36]:
razao_especialista_junior = round(
    senioridade_salario.set_index("senioridade").loc["Especialista/Staff+", "salario_medio"]
    / senioridade_salario.set_index("senioridade").loc["Júnior", "salario_medio"],
    1,
)
print(f"Especialista/Staff+ ganha {razao_especialista_junior}x mais que Júnior em salário médio.")

Especialista/Staff+ ganha 4.9x mais que Júnior em salário médio.


### 6.3 Diferenças por modelo de trabalho

In [37]:
salario_modelo = (
    perfil.groupby("modalidade_simples")["salario_medio"].agg(salario_medio="mean", n="count").dropna()
)
salario_modelo["salario_medio"] = salario_modelo["salario_medio"].round(0)
df_salario_modelo = salario_modelo.reset_index().merge(
    df_modelo[["modelo", "%"]], left_on="modalidade_simples", right_on="modelo"
).drop(columns="modalidade_simples").sort_values("salario_medio", ascending=False)
df_salario_modelo

,salario_medio,n,modelo,%
2,"12,522.0",5702,Remoto,44.4
0,"12,366.0",4886,Híbrido,38.1
1,"7,874.0",2252,Presencial,17.5


In [38]:
evolucao_modelo = {}
for ano, grupo in perfil.dropna(subset=["modalidade_simples"]).groupby("ano_pesquisa"):
    evolucao_modelo[int(ano)] = largest_remainder_pct(grupo["modalidade_simples"].value_counts().to_dict())
df_evolucao_modelo = pd.DataFrame(evolucao_modelo).T
df_evolucao_modelo.index.name = "ano"
print("[checagem] soma por ano:")
print(df_evolucao_modelo.sum(axis=1))
df_evolucao_modelo

[checagem] soma por ano:
ano
2023   100.0
2024   100.0
2025   100.0
dtype: float64


,Remoto,Híbrido,Presencial
ano,,,
2023,46.3,37.1,16.6
2024,45.7,38.0,16.3
2025,39.7,39.5,20.8


## Pergunta 7 — Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e IA?

Todos os itens abaixo são perguntas de múltipla escolha (percentuais não somam 100%).

### 7.1 O que atrai talento (critério de escolha de emprego)

In [39]:
criterio = satisfacao[satisfacao["categoria"] == "criterio_escolha"]
base_criterio = criterio["respondente_id"].nunique()
vc_criterio = criterio["item"].value_counts().head(6)
df_criterio = vc_criterio.rename_axis("item").reset_index(name="n")
df_criterio["% (base=" + str(base_criterio) + ")"] = (df_criterio["n"] / base_criterio * 100).round(1)
df_criterio

,item,n,% (base=12791)
0,Remuneração/Salário,10470,81.9
1,Flexibilidade de trabalho remoto,7390,57.8
2,Plano de carreira,4172,32.6
3,Aprendizado/referências na área,3103,24.3
4,Benefícios,3101,24.2
5,Ambiente e clima de trabalho,2599,20.3


### 7.2 O que gera insatisfação (oportunidade de retenção)

In [40]:
insatisfacao = satisfacao[satisfacao["categoria"] == "motivo_insatisfacao"]
base_insat = insatisfacao["respondente_id"].nunique()
vc_insat = insatisfacao["item"].value_counts().head(6)
df_insatisfacao = vc_insat.rename_axis("item").reset_index(name="n")
df_insatisfacao["% (base=" + str(base_insat) + ")"] = (df_insatisfacao["n"] / base_insat * 100).round(1)
df_insatisfacao

,item,n,% (base=3762)
0,Salário não corresponde ao mercado,1698,45.1
1,Falta de oportunidade de crescimento,1519,40.4
2,Falta de maturidade analítica,1219,32.4
3,Falta de aprendizado/referências,819,21.8
4,Quer trabalhar em outra área,787,20.9
5,Poucos benefícios,680,18.1


### 7.3 Principais desafios de quem já gerencia dados

In [41]:
desafio = gestao[gestao["categoria"] == "desafio"]
base_desafio = desafio["respondente_id"].nunique()
vc_desafio = desafio["item"].value_counts().head(6)
df_desafio = vc_desafio.rename_axis("item").reset_index(name="n")
df_desafio["% (base=" + str(base_desafio) + ")"] = (df_desafio["n"] / base_desafio * 100).round(1)
df_desafio

,item,n,% (base=2552)
0,Gerenciar expectativa do negócio,902,35.3
1,Dividir tempo técnico/gestão,813,31.9
2,Gerar valor pro negócio,704,27.6
3,Qualidade/confiabilidade dos dados,694,27.2
4,Levar inovação pra empresa,575,22.5
5,Projetos multidisciplinares,573,22.5


### 7.4 Barreiras específicas à adoção de IA

Já detalhado na Seção 5.4 — reaproveitado aqui para referência cruzada.

In [42]:
df_motivos_desuso

,item,n,% (base=2290)
0,Falta de expertise/recursos,842,36.8
1,Falta de casos de uso claros,825,36.0
2,Dados não prontos pra IA,757,33.1
3,Preocupação com segurança/privacidade,739,32.3
4,ROI não comprovado,542,23.7
5,Falta de confiabilidade (alucinação),402,17.6
6,Incerteza regulatória,301,13.1
7,Preocupação com propriedade intelectual,298,13.0


## Nota metodológica e limitações

- Todos os números acima vêm das tabelas da camada **gold** do projeto
  (`tbl_fato_perfil_profissional`, `tbl_fato_tecnologias`, `tbl_fato_ia_generativa`,
  `tbl_fato_gestao`, `tbl_fato_satisfacao`, `tbl_dimensao_faixa_salarial`), exceto o dado de
  carreira prejudicada por gênero/raça (Seção 3.3), extraído da camada **silver** por não
  estar materializado na gold.
- Salário médio é o ponto médio (`valor_medio`) de cada faixa salarial declarada.
- Perguntas de múltipla escolha têm percentuais calculados sobre respondentes únicos de cada
  bloco (indicado em cada tabela), não sobre o total de 14.002.
- Variáveis categóricas que representam uma partição completa da base (gênero, senioridade,
  região, raça/cor, setor, porte, modelo de trabalho, PCD, prioridade de IA) usam o método do
  maior resto para garantir soma == 100,0% exatamente.
- Número de vagas em aberto e turnover não estão disponíveis no dataset.
- **Divergência de dados conhecida:** 2 registros de `num_funcionarios` têm o valor
  "de 501 a 100" (fora das 8 faixas oficiais do questionário) — foram excluídos da tabela de
  porte de empresas (Seção 1.2), e essa exclusão é reportada explicitamente acima, não de
  forma silenciosa.

## Checagem final — consistência com a apresentação e a documentação

Esta célula recalcula, a partir do zero (variáveis independentes, sem reaproveitar nada
calculado acima), um pequeno conjunto de números-chave e os compara com os valores publicados
na apresentação HTML e na documentação do projeto. Qualquer `AssertionError` aqui indica uma
divergência real entre este notebook e os outros dois artefatos — e deveria ser investigada
antes de anexar o notebook ao projeto.

In [43]:
valores_publicados = {
    "genero_masculino_pct": 76.0,
    "genero_feminino_pct": 23.5,
    "senioridade_senior_pct": 37.9,
    "senioridade_pleno_pct": 34.8,
    "senioridade_junior_pct": 23.9,
    "senioridade_especialista_pct": 3.4,
    "cargo_base_respondentes": 10173,
    "analista_cientista_pct": 41.9,
    "ia_usuarios_reais_base": 8514,
    "ia_uso_gratuito_pct": 58.6,
    "setor_industria_pct": 6.9,
    "porte_51_100_pct": 7.1,
}

checagens = {
    "genero_masculino_pct": pct_genero_dict["Masculino"],
    "genero_feminino_pct": pct_genero_dict["Feminino"],
    "senioridade_senior_pct": pct_senioridade_dict["Sênior"],
    "senioridade_pleno_pct": pct_senioridade_dict["Pleno"],
    "senioridade_junior_pct": pct_senioridade_dict["Júnior"],
    "senioridade_especialista_pct": pct_senioridade_dict["Especialista/Staff+"],
    "cargo_base_respondentes": BASE_CARGO,
    "analista_cientista_pct": pct_analista_cientista,
    "ia_usuarios_reais_base": BASE_USUARIOS_REAIS,
    "ia_uso_gratuito_pct": float(df_uso_pessoal.loc[df_uso_pessoal["item"] == "Uso solução gratuita", "% (entre usuários reais)"].iloc[0]),
    "setor_industria_pct": pct_setor_dict["Indústria"],
    "porte_51_100_pct": pct_porte_dict["de 51 a 100"],
}

print(f"{'métrica':40s} {'notebook':>10s} {'publicado':>10s}  status")
tudo_ok = True
for chave, esperado in valores_publicados.items():
    obtido = checagens[chave]
    ok = abs(obtido - esperado) < 0.05
    tudo_ok &= ok
    print(f"{chave:40s} {obtido:>10} {esperado:>10}  {'OK' if ok else 'DIVERGE'}")

assert tudo_ok, "Há divergência entre este notebook e os valores publicados na apresentação/documentação."
print("\nTodos os valores-chave batem com a apresentação HTML e a documentação. ✔")

métrica                                    notebook  publicado  status
genero_masculino_pct                           76.0       76.0  OK
genero_feminino_pct                            23.5       23.5  OK
senioridade_senior_pct                         37.9       37.9  OK
senioridade_pleno_pct                          34.8       34.8  OK
senioridade_junior_pct                         23.9       23.9  OK
senioridade_especialista_pct                    3.4        3.4  OK
cargo_base_respondentes                       10173      10173  OK
analista_cientista_pct                         41.9       41.9  OK
ia_usuarios_reais_base                         8514       8514  OK
ia_uso_gratuito_pct                            58.6       58.6  OK
setor_industria_pct                             6.9        6.9  OK
porte_51_100_pct                                7.1        7.1  OK

Todos os valores-chave batem com a apresentação HTML e a documentação. ✔
